In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, recall_score
from catboost import CatBoostClassifier, Pool
import os
from pymatgen.core import Composition, Element

In [2]:
data = pd.read_csv('./data/oxidase_data.csv')

In [3]:
positive = data[data['label'] == 1]
negative = data[data['label'] == 0]

In [4]:
N_POSITIVE = len(positive) // 2
test_positive = positive.sample(n=N_POSITIVE, random_state=42)
remove_test_positive = positive.drop(test_positive.index).reset_index(drop=True)
test_positive = test_positive.reset_index(drop=True)

In [5]:
features = [
    'pearson',
    'space',
    'volume',
    'density',
    
    'band_gap',
    'efermi',
    'formation_energy_per_atom',
    
    'MagpieData mean NsValence',
    'MagpieData mean NpValence',
    'MagpieData mean NdValence',
    'MagpieData mean NfValence',
    
    'MagpieData mean NUnfilled',
    
    'MagpieData mean CovalentRadius',
    'MagpieData mean AtomicWeight',
    
    'MagpieData mean Electronegativity',
    'MagpieData mean GSmagmom',
    
    'MagpieData mean Row',
    'MagpieData mean Column'
]

In [6]:
X_test_positive = test_positive[features]
y_test_positive = test_positive['label'] 

In [7]:
n_positive = remove_test_positive.shape[0]
n_negative = n_positive

In [8]:
categorical_features = ['pearson'] if 'pearson' in remove_test_positive[features].columns else []

In [9]:
import numpy as np


N_RUNS = 500
ITERATIONS = 300
LEARNING_RATE = 0.05
DEPTH = 8
LOSS_FUNCTION = 'Logloss'
EVAL_METRIC = 'AUC'
RANDOM_SEED = 0
VERBOSE = 0

recall_per_run = []
accuracy_per_run = []
f1_per_run = []
auroc_per_run = []
auprc_per_run = []
model_per_run = []

for run in range(0, N_RUNS):
    X_train_negative = negative.sample(n=n_negative, random_state=run)
    neg_for_test_pool = negative.drop(X_train_negative.index)
    X_test_negative = neg_for_test_pool.sample(n=len(X_test_positive), random_state=run)[features]
    
    all_df = pd.concat([remove_test_positive, X_train_negative], axis=0).reset_index(drop=True)
    X_all = all_df[features]
    y_all = all_df['label']
    
    train_pool = Pool(X_all, y_all, cat_features=categorical_features)
    
    model = CatBoostClassifier(
        iterations=ITERATIONS,
        learning_rate=LEARNING_RATE,
        depth=DEPTH,
        loss_function=LOSS_FUNCTION,
        eval_metric=EVAL_METRIC,
        random_seed=RANDOM_SEED,
        verbose=VERBOSE
    )
    model.fit(train_pool)
    model_per_run.append(model)
    
    test_positive_pool = Pool(X_test_positive, cat_features=categorical_features)
    test_negative_pool = Pool(X_test_negative, cat_features=categorical_features)
    
    y_pred_proba_positive = model.predict_proba(test_positive_pool)[:, 1]
    y_pred_proba_negative = model.predict_proba(test_negative_pool)[:, 1]
    
    y_pred_positive = model.predict(test_positive_pool)
    y_pred_negative = model.predict(test_negative_pool)
    
    y_true = np.concatenate([y_test_positive, np.zeros(len(X_test_negative))])
    y_pred = np.concatenate([y_pred_positive, y_pred_negative])
    y_pred_proba = np.concatenate([y_pred_proba_positive, y_pred_proba_negative])
    
    recall = recall_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auroc = roc_auc_score(y_true, y_pred_proba)
    auprc = average_precision_score(y_true, y_pred_proba)
    
    recall_per_run.append(recall)
    accuracy_per_run.append(accuracy)
    f1_per_run.append(f1)
    auroc_per_run.append(auroc)
    auprc_per_run.append(auprc)
    
    print(f"Run {run}: Recall={recall:.3f}, Accuracy={accuracy:.3f}, F1={f1:.3f}, AUROC={auroc:.3f}, AUPRC={auprc:.3f}")

Run 0: Recall=0.875, Accuracy=0.825, F1=0.833, AUROC=0.924, AUPRC=0.926
Run 1: Recall=0.900, Accuracy=0.863, F1=0.867, AUROC=0.920, AUPRC=0.912
Run 2: Recall=0.925, Accuracy=0.838, F1=0.851, AUROC=0.944, AUPRC=0.952
Run 3: Recall=0.800, Accuracy=0.825, F1=0.821, AUROC=0.913, AUPRC=0.900
Run 4: Recall=0.900, Accuracy=0.800, F1=0.818, AUROC=0.902, AUPRC=0.904
Run 5: Recall=0.900, Accuracy=0.875, F1=0.878, AUROC=0.909, AUPRC=0.924
Run 6: Recall=0.950, Accuracy=0.875, F1=0.884, AUROC=0.953, AUPRC=0.958
Run 7: Recall=0.900, Accuracy=0.900, F1=0.900, AUROC=0.955, AUPRC=0.956
Run 8: Recall=0.975, Accuracy=0.912, F1=0.918, AUROC=0.971, AUPRC=0.969
Run 9: Recall=0.775, Accuracy=0.812, F1=0.805, AUROC=0.928, AUPRC=0.934
Run 10: Recall=0.825, Accuracy=0.850, F1=0.846, AUROC=0.938, AUPRC=0.937
Run 11: Recall=0.900, Accuracy=0.863, F1=0.867, AUROC=0.917, AUPRC=0.931
Run 12: Recall=0.875, Accuracy=0.900, F1=0.897, AUROC=0.954, AUPRC=0.962
Run 13: Recall=0.975, Accuracy=0.925, F1=0.929, AUROC=0.988, 

In [10]:
recall_df = pd.DataFrame({
    "run": range(1, len(recall_per_run) + 1),
    "recall": pd.Series(recall_per_run, dtype=float),
    "accuracy": pd.Series(accuracy_per_run, dtype=float),
    "f1": pd.Series(f1_per_run, dtype=float),
    "auroc": pd.Series(auroc_per_run, dtype=float),
    "auprc": pd.Series(auprc_per_run, dtype=float)
})

In [23]:
recall_df

,run,recall,accuracy,f1,auroc,auprc
0,1,0.875,0.8250,0.833333,0.923750,0.926323
1,2,0.900,0.8625,0.867470,0.920000,0.911681
2,3,0.925,0.8375,0.850575,0.943750,0.951538
3,4,0.800,0.8250,0.820513,0.913125,0.899877
4,5,0.900,0.8000,0.818182,0.901875,0.904422
...,...,...,...,...,...,...
495,496,0.850,0.8250,0.829268,0.932500,0.925112
496,497,0.900,0.8250,0.837209,0.894375,0.909402
497,498,0.925,0.9250,0.925000,0.966875,0.970416
498,499,0.925,0.8375,0.850575,0.916250,0.904209


In [22]:
recall_df.to_csv('./recall.csv')

In [12]:
arr = np.array(recall_per_run)
topk = 50
top_idx = arr.argsort()[-topk:][::-1]
top_runs = [i + 1 for i in top_idx]

top_models = [model_per_run[i] for i in top_idx]

print("The best 50 models ：")
for run, recall in zip(top_runs, arr[top_idx]):
    print(f"Run {run} : Recall={recall:.4f}")

The best 50 models ：
Run 488 : Recall=1.0000
Run 161 : Recall=0.9750
Run 134 : Recall=0.9750
Run 9 : Recall=0.9750
Run 14 : Recall=0.9750
Run 432 : Recall=0.9750
Run 179 : Recall=0.9500
Run 417 : Recall=0.9500
Run 66 : Recall=0.9500
Run 170 : Recall=0.9500
Run 74 : Recall=0.9500
Run 106 : Recall=0.9500
Run 391 : Recall=0.9500
Run 425 : Recall=0.9500
Run 206 : Recall=0.9500
Run 88 : Recall=0.9500
Run 319 : Recall=0.9500
Run 256 : Recall=0.9500
Run 254 : Recall=0.9500
Run 343 : Recall=0.9500
Run 323 : Recall=0.9500
Run 490 : Recall=0.9500
Run 226 : Recall=0.9500
Run 489 : Recall=0.9500
Run 480 : Recall=0.9500
Run 121 : Recall=0.9500
Run 7 : Recall=0.9500
Run 352 : Recall=0.9500
Run 362 : Recall=0.9250
Run 169 : Recall=0.9250
Run 307 : Recall=0.9250
Run 356 : Recall=0.9250
Run 404 : Recall=0.9250
Run 71 : Recall=0.9250
Run 164 : Recall=0.9250
Run 297 : Recall=0.9250
Run 292 : Recall=0.9250
Run 119 : Recall=0.9250
Run 80 : Recall=0.9250
Run 416 : Recall=0.9250
Run 418 : Recall=0.9250
Run 6

In [13]:
save_dir = "./model"
os.makedirs(save_dir, exist_ok=True)

for i, model in enumerate(top_models, start=1):
    model_path = os.path.join(save_dir, f"model_{i}.cbm")
    model.save_model(model_path, format="cbm")

In [14]:
arr = np.array(recall_per_run)
top_idx = arr.argsort()[-50:][::-1]

loss_dict = {}

for idx in top_idx:
    model = model_per_run[idx]
    evals_result = model.get_evals_result()
    losses = evals_result["learn"].get(LOSS_FUNCTION, [])
    col_name = f"run_{idx+1}_loss"
    loss_dict[col_name] = losses

loss_df = pd.DataFrame(loss_dict)
loss_df.insert(0, "iteration", range(1, len(loss_df)+1))

In [24]:
loss_df

,iteration,run_488_loss,run_161_loss,run_134_loss,run_9_loss,run_14_loss,run_432_loss,run_179_loss,run_417_loss,run_66_loss,...,run_418_loss,run_60_loss,run_132_loss,run_59_loss,run_340_loss,run_174_loss,run_79_loss,run_83_loss,run_81_loss,run_113_loss
0,1,0.658740,0.660880,0.652656,0.655323,0.653611,0.656891,0.653128,0.660525,0.658634,...,0.653430,0.659754,0.655485,0.654817,0.657571,0.657021,0.655405,0.654528,0.653098,0.653665
1,2,0.621106,0.627527,0.619388,0.629178,0.617868,0.629005,0.616773,0.634337,0.625056,...,0.619156,0.611732,0.625697,0.610515,0.623305,0.623022,0.617830,0.617231,0.621587,0.625869
2,3,0.591784,0.597835,0.589006,0.598709,0.586646,0.590167,0.587316,0.603863,0.590939,...,0.590664,0.575379,0.600765,0.573516,0.593711,0.593973,0.589891,0.584818,0.592064,0.596222
3,4,0.556412,0.568952,0.558896,0.574964,0.558028,0.558338,0.557081,0.576455,0.561732,...,0.555610,0.545276,0.576781,0.547879,0.567717,0.569530,0.564178,0.558819,0.568393,0.572691
4,5,0.528841,0.540349,0.535775,0.548138,0.526885,0.532032,0.531209,0.550478,0.530319,...,0.529476,0.523117,0.555483,0.520552,0.543922,0.546380,0.532432,0.537564,0.537564,0.546114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,0.009627,0.012109,0.013288,0.009721,0.010738,0.008991,0.011370,0.011423,0.012782,...,0.009218,0.007390,0.009723,0.011577,0.012851,0.008970,0.008266,0.010455,0.010729,0.010484
296,297,0.009594,0.012060,0.013255,0.009674,0.010699,0.008945,0.011325,0.011374,0.012741,...,0.009182,0.007362,0.009686,0.011540,0.012785,0.008922,0.008222,0.010422,0.010683,0.010451
297,298,0.009561,0.012011,0.013205,0.009631,0.010660,0.008916,0.011283,0.011334,0.012697,...,0.009142,0.007339,0.009638,0.011505,0.012720,0.008895,0.008196,0.010389,0.010609,0.010406
298,299,0.009520,0.011963,0.013172,0.009564,0.010620,0.008886,0.011240,0.011285,0.012643,...,0.009106,0.007310,0.009603,0.011465,0.012673,0.008868,0.008153,0.010313,0.010565,0.010374


In [16]:
mp_predict = negative[['formula']].copy()

for i, model in enumerate(top_models, start=1):
    col_name = f"model_{i}_score"
    mp_predict[col_name] = model.predict_proba(negative[features])[:, 1]
    
model_score_cols = [f"model_{i}_score" for i in range(1, len(top_models)+1)]
mp_predict["avg_score"] = mp_predict[model_score_cols].mean(axis=1)
mp_predict["std_score"] = mp_predict[model_score_cols].std(axis=1)
mp_predict = mp_predict.sort_values(by="avg_score", ascending=False)

In [17]:
mp_predict.to_csv('./data/oxidase_predict_materials_project.csv', index=False)

In [18]:
def contains_transition_metal(formula: str) -> bool:
    comp = Composition(formula)
    for el in comp.elements:
        if Element(el).is_transition_metal:
            return True
    return False

mp_predict["has_TM"] = mp_predict["formula"].apply(contains_transition_metal)

bins = np.linspace(0, 1, 11)
labels = [f"{bins[i]:.1f} - {bins[i+1]:.1f}" for i in range(len(bins)-1)]
mp_predict["avg_bin"] = pd.cut(
    mp_predict["avg_score"].clip(0,1),
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

bin_stats = (
    mp_predict
    .groupby("avg_bin")
    .agg(total=("formula", "count"),
         tm_count=("has_TM", "sum"))
)
bin_stats["tm_ratio"] = bin_stats["tm_count"] / bin_stats["total"]

C:\Users\yangmutian\AppData\Local\Temp\ipykernel_4264\4046836825.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("avg_bin")


In [19]:
bin_stats

,total,tm_count,tm_ratio
avg_bin,,,
0.0 - 0.1,28881,21580,0.747204
0.1 - 0.2,24591,16365,0.665487
0.2 - 0.3,15947,11174,0.700696
0.3 - 0.4,10093,7597,0.752700
0.4 - 0.5,7418,5875,0.791992
0.5 - 0.6,5615,4510,0.803206
0.6 - 0.7,4363,3584,0.821453
0.7 - 0.8,3344,2680,0.801435
0.8 - 0.9,2526,2187,0.865796


In [20]:
mp_predict.has_TM.mean()

0.7388578789179906